# Spaceship Titanic 本番

**モード:** 本番（`competition/`）＝ **協力戦**。AI も道具として駆使して、実際にリーダーボードを上げにいく。
方向（何を試すか・いつ提出するか）は自分が決め、コードは Claude と一緒に書いて回す。**この器のコードセルは空＝自分で書く。**

**評価指標:** accuracy（正解率）　／　**タスク:** 二値分類（`Transported` を True / False で当てる）

**現在地:** 未提出（これから）。まずは提出パイプラインを一度通し、そこから手元CVを物差しに上げていく。

**Titanic で効いた型（そのまま持ち込む）:**
①正直な物差し（seed複数平均CV）を先に作る → ②過学習gapを縮める → ③本当に新しい情報を足す。
今回その「新しい情報」の主戦場が **特徴量エンジニアリング**（Cabin分解・グループ特徴量）。ここが今日の本筋。

本番の流れ:
```
train.csv (Transported有) ──fit──▶ モデル ──predict──▶ test.csv (Transported無)
                                                         │
                                                         ▼
                                   submissions/submission.csv  (PassengerId, Transported)
                                                         │  kaggle competitions submit
                                                         ▼
                                                  public リーダーボード
```
> このコンペは合成データ＋テスト正解は非公開＝**答えを覗けない本物の勝負**（正規の上位は accuracy ≈ 0.80〜0.81 が天井）。

## 📋 使うデータ：宇宙船 Spaceship Titanic の「乗客記録」

西暦2912年、宇宙船 Spaceship Titanic が時空の異常に衝突。乗客の約半数が **別次元へ転送(Transported)** された。
やること = **乗客のプロフィール（出身惑星・冷凍睡眠・部屋・船内消費…）から「転送された(True) / されなかった(False)」を当てる。**
（構造は Titanic とまったく同じ二値分類。当てる対象が「生死」→「転送の有無」に変わっただけ）

### 2つのファイル（ここが本番の肝）
| ファイル | 行数(目安) | Transported(正解) | 役割 |
|---|---|---|---|
| `train.csv` | 約8,700 | **あり** | 「どんな人が転送されたか」を勉強させる過去問 |
| `test.csv`  | 約4,300 | **なし** | この人たちを予測して Kaggle に提出する本番問題 |

### 列の意味
| 列 | 意味 | メモ |
|---|---|---|
| **Transported** | 🎯 **正解（当てたいもの）** True=転送 / False=残留 | これを予測する |
| PassengerId | `gggg_pp` 形式。gggg=**同行グループ番号**, pp=その中の番号 | 同じ gggg は家族/連れの可能性 |
| HomePlanet | 出身惑星（Earth / Europa / Mars） | |
| CryoSleep | 航海中ずっと**冷凍睡眠**だったか（True/False） | 睡眠中は船内消費が0になるはず |
| Cabin | 部屋番号 `deck/num/side`（例 `B/0/P`） | **`/`で deck・num・side に割れる**。side は P=Port / S=Starboard |
| Destination | 目的地の惑星 | |
| Age | 年齢 | |
| VIP | VIP料金を払ったか | |
| RoomService / FoodCourt / ShoppingMall / Spa / VRDeck | 各施設での**消費額**（5列） | 合計＝総消費。0が多い人＝冷凍睡眠？ |
| Name | 氏名（姓＝家族の手がかり） | |

> 転送はランダムじゃなく **かたより** がある（例: 冷凍睡眠者・特定の deck・消費パターンで転送率が違う）。
> そのかたよりを掴むのが EDA。そして「1人の属性」だけでなく **グループ(gggg)/家族で揃う** かもしれない —— ここが Titanic の FamilySurv と同じ匂いのする狙い目。

## 1. 読み込み・EDA

In [ ]:
# ============================================================
# 節1: データを読み込んで「どんなデータか」をつかむ（EDA）
#   狙い = 節2の前処理・特徴量を決めるための材料集め
#   見るとよい4点: 形 / 中身(head) / 欠損 / 目的変数(Transported)の偏り
# ✋予想: train と test の行数・列数はそれぞれいくつ？ test は何列少ない？
#         欠損が多そうな列は？ Transported の True/False 比はだいたい半々？
# ============================================================
# ここから自分で書く


In [ ]:
# ============================================================
# 節1.5: 「その列、本当に転送のヒント？」を転送率で確かめる（EDA）
#   なぜ: 特徴量にする前に「Transported と差が出る列か」を先に見る。
#         全体の転送率(基準線)より大きくズレる列ほど強い手がかり。
#   道具: df.groupby(列)["Transported"].mean()
# ✋予想: CryoSleep / HomePlanet / Cabin の side(P/S) ―― どれが一番くっきり差が出ると思う？
# ============================================================
# ここから自分で書く


## 2. 前処理・特徴量づくり（今日の本筋）

In [ ]:
# ============================================================
# 節2: 前処理 + 特徴量づくり
#   リーク回避の定石: train+test を縦結合し『特徴量だけ』から作る（正解 Transported は使わない）。
#   候補(自分で取捨選択する):
#     - 欠損補完（数値=中央値 など / カテゴリ=最頻 or "Unknown"）
#     - Cabin を deck / num / side に分解
#     - 消費5列の合計 TotalSpend、"全部0か" フラグ（冷凍睡眠と絡む？）
#     - PassengerId の gggg からグループサイズ など
# ✋予想: 上のうち「効きそう」な順に3つ挙げてから作ってみて。あとで手元CVで答え合わせする。
# ============================================================
# ここから自分で書く


## 3. モデル（LightGBM 単体）＋ 正直な手元CV

In [ ]:
# ============================================================
# 節3: モデル(LightGBM) + 正直な手元CV
#   物差し: StratifiedKFold + seed複数平均 = まぐれ(当たりseed)に騙されない（Titanicでやった型）。
#   まずデフォルト寄りで1本通して「正直CV」を出す → そこからチューニング/特徴量で上げる。
# ✋予想: 何もせず多数派クラスに全部寄せたら accuracy は約いくつ？ 最初のCVはそれをどれだけ上回る？
# ============================================================
# ここから自分で書く


## 4. test を予測 → submission.csv（PassengerId, Transported / index=False）

In [ ]:
# ============================================================
# 節4: test を予測して提出ファイルを作る
#   提出フォーマット: 2列 [PassengerId, Transported]。Transported は True/False（bool）。
#   保存先: submissions/submission.csv を index=False で書く。
#   モデルは全 train で学習し直す（CVより多いデータで本番モデルにする）。
# ✋予想: 予測の True/False 比は、train の比(≈半々)に近くなるはず？ ズレてたら何かおかしいサイン。
# ============================================================
# ここから自分で書く


## 5. 提出（自分で）

```
kaggle competitions submit -c spaceship-titanic -f submissions/submission.csv -m "メッセージ"
```
提出したら public スコアを持ってきて。予想と照らして「なぜその点か」を一緒に開ける。